# E010 nonlinear candidate coverage and selector regret
Private, internet-disabled, CPU-only full validation. Attach the exact sealed E010 input dataset version and the ROGII competition data. The notebook caps all native CPU pools at two threads, imports the exact bundled `src/` package, and fails closed on any identity or control mismatch.


In [ ]:
CONTRACT = {
  "BUNDLE_BYTES": 549143874,
  "BUNDLE_FILENAME": "rogii-e010-inputs-v2.zip.bin",
  "BUNDLE_SHA256": "e26884fcfd49b41514f78b32cb7e46d87a87fb776791c41022ac912039864cad",
  "CONFIG_SHA256": "82fe4f8b453a17a4b4f66a29733e0909dd0881f9682984e52bd004fce07686cd",
  "DATASET_REF": "ashok205/rogii-e010-selector-inputs",
  "DATASET_VERSION": 2,
  "DATA_SIGNATURE": "6ebe65b403f80fefd97dcd7bbfce7314252e779a1837c97364cf55761590fe77",
  "EXPECTED_WELLS": 773,
  "INPUT_RECEIPT_FILENAME": "e010-input-receipt-v2.json",
  "INPUT_RECEIPT_SHA256": "6460b3a5e04c8b4a8c4357a3dfdda9b0c0aecb6ff1c258002fe9ae6fe103dced",
  "OUTPUT_MANIFEST_FILENAME": "e010-output-manifest.json",
  "RESULT_ARCHIVE_FILENAME": "rogii-e010-results-v2.zip",
  "RUN_RECEIPT_FILENAME": "e010-run-receipt.json",
  "SOURCE_COMMIT": "49ca3995269638026561c5593ad2628f8f08a4e9",
  "THREAD_LIMIT": 2
}


In [ ]:
import datetime as _dt
import hashlib
import importlib
import json
import os
import platform
import shutil
import sys
import zipfile
from pathlib import Path, PurePosixPath

for _name in (
    "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS",
):
    os.environ[_name] = str(CONTRACT["THREAD_LIMIT"])
os.environ["JOBLIB_MULTIPROCESSING"] = "0"

WORKING = Path("/kaggle/working")
RUNTIME_ROOT = WORKING / "rogii-e010-runtime"
RESULT_DIR = WORKING / "e010-results"
ARTIFACT_DIR = WORKING / "e010-artifacts"
PREFLIGHT_PATH = WORKING / "e010-preflight.json"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(16 * 1024 * 1024)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def _safe(name: str) -> str:
    raw = str(name).replace("\\", "/")
    raw_parts = raw.split("/")
    value = PurePosixPath(raw)
    if value.is_absolute() or not raw or any(part in {"", ".", ".."} for part in raw_parts) or not value.parts:
        raise RuntimeError(f"unsafe archive member {name!r}")
    return value.as_posix()

bundle_matches = sorted(Path("/kaggle/input").rglob(CONTRACT["BUNDLE_FILENAME"]))
receipt_matches = sorted(Path("/kaggle/input").rglob(CONTRACT["INPUT_RECEIPT_FILENAME"]))
if len(bundle_matches) != 1 or len(receipt_matches) != 1:
    raise RuntimeError({"bundle_matches": [str(p) for p in bundle_matches], "receipt_matches": [str(p) for p in receipt_matches]})
bundle_path = bundle_matches[0]
input_receipt_path = receipt_matches[0]
if bundle_path.stat().st_size != CONTRACT["BUNDLE_BYTES"] or _sha256(bundle_path) != CONTRACT["BUNDLE_SHA256"]:
    raise RuntimeError("sealed E010 bundle identity mismatch")
if _sha256(input_receipt_path) != CONTRACT["INPUT_RECEIPT_SHA256"]:
    raise RuntimeError("E010 input receipt identity mismatch")
input_receipt = json.loads(input_receipt_path.read_text(encoding="utf-8"))
if input_receipt["dataset"]["ref"] != CONTRACT["DATASET_REF"] or int(input_receipt["dataset"]["version"]) != int(CONTRACT["DATASET_VERSION"]):
    raise RuntimeError("attached E010 dataset ref/version differs from notebook contract")

if RUNTIME_ROOT.exists():
    shutil.rmtree(RUNTIME_ROOT)
RUNTIME_ROOT.mkdir(parents=True)
with zipfile.ZipFile(bundle_path) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)):
        raise RuntimeError("duplicate E010 archive members")
    for name in names:
        _safe(name)
    archive.extractall(RUNTIME_ROOT)
manifest = json.loads((RUNTIME_ROOT / "bundle_manifest.json").read_text(encoding="utf-8"))
if manifest["source_commit"] != CONTRACT["SOURCE_COMMIT"] or manifest["config_sha256"] != CONTRACT["CONFIG_SHA256"]:
    raise RuntimeError("E010 bundle source/config contract mismatch")
for entry in manifest["files"]:
    path = RUNTIME_ROOT / _safe(entry["path"])
    if not path.is_file() or path.stat().st_size != int(entry["bytes"]) or _sha256(path) != entry["sha256"]:
        raise RuntimeError(f"E010 extracted member mismatch: {entry['path']}")

# Kaggle interactive sessions retain imported modules across notebook re-imports.
# Remove every prior ROGII module so this run can only use the hash-verified
# package extracted above, never a stale module from an earlier failed notebook.
for _module_name in tuple(sys.modules):
    if _module_name == "rogii_validation" or _module_name.startswith("rogii_validation."):
        del sys.modules[_module_name]
importlib.invalidate_caches()
sys.path.insert(0, str(RUNTIME_ROOT / "src"))
from rogii_validation.harness import scan_profiles

preferred_roots = (
    Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
    Path("/kaggle/input/rogii-wellbore-geology-prediction"),
)
COMPETITION_ROOT = next(
    (
        root
        for root in preferred_roots
        if (root / "train").is_dir() and (root / "sample_submission.csv").is_file()
    ),
    None,
)
if COMPETITION_ROOT is None:
    fallback_roots = sorted(
        sample.parent
        for sample in Path("/kaggle/input").rglob("sample_submission.csv")
        if (sample.parent / "train").is_dir()
    )
    if len(fallback_roots) != 1:
        raise RuntimeError({"competition_roots_found": [str(root) for root in fallback_roots]})
    COMPETITION_ROOT = fallback_roots[0]
TRAIN_DIR = COMPETITION_ROOT / "train"
_, DATA_PROFILE = scan_profiles(TRAIN_DIR)
if int(DATA_PROFILE["well_count"]) != int(CONTRACT["EXPECTED_WELLS"]) or DATA_PROFILE["data_signature"] != CONTRACT["DATA_SIGNATURE"]:
    raise RuntimeError({
        "competition_root": str(COMPETITION_ROOT),
        "expected_wells": CONTRACT["EXPECTED_WELLS"],
        "expected_signature": CONTRACT["DATA_SIGNATURE"],
        "actual_profile": DATA_PROFILE,
    })

preflight = {
    "schema_version": 1,
    "experiment_id": "E010",
    "status": "PASS",
    "checked_at_utc": _dt.datetime.now(_dt.timezone.utc).isoformat(),
    "source_commit": CONTRACT["SOURCE_COMMIT"],
    "config_sha256": CONTRACT["CONFIG_SHA256"],
    "bundle": {"path": str(bundle_path), "bytes": bundle_path.stat().st_size, "sha256": _sha256(bundle_path)},
    "input_receipt": {"path": str(input_receipt_path), "sha256": _sha256(input_receipt_path)},
    "dataset": {"ref": CONTRACT["DATASET_REF"], "version": CONTRACT["DATASET_VERSION"]},
    "competition_root": str(COMPETITION_ROOT),
    "data_profile": DATA_PROFILE,
    "runtime": {
        "python": sys.version,
        "platform": platform.platform(),
        "logical_cpus": os.cpu_count(),
        "thread_limit": CONTRACT["THREAD_LIMIT"],
        "thread_environment": {name: os.environ.get(name) for name in (
            "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
            "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS",
        )},
        "internet_expected": False,
        "accelerator_expected": "CPU",
    },
}
PREFLIGHT_PATH.write_text(json.dumps(preflight, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(preflight, indent=2, sort_keys=True))


In [ ]:
import datetime as _dt
import json
import traceback
from pathlib import Path

import numpy as np
import sklearn
from threadpoolctl import threadpool_info, threadpool_limits
from rogii_validation.nonlinear_selector import run_e010

started_at = _dt.datetime.now(_dt.timezone.utc)
config_path = RUNTIME_ROOT / "experiments/E010/config.json"
config = json.loads(config_path.read_text(encoding="utf-8"))
try:
    with threadpool_limits(limits=CONTRACT["THREAD_LIMIT"]):
        active_pools_before = threadpool_info()
        if any(int(item.get("num_threads", 0) or 0) > int(CONTRACT["THREAD_LIMIT"]) for item in active_pools_before):
            raise RuntimeError({"thread_pool_limit_failed": active_pools_before})
        summary = run_e010(
            root=RUNTIME_ROOT,
            train_dir=TRAIN_DIR,
            output_dir=RESULT_DIR,
            artifact_dir=ARTIFACT_DIR,
            config=config,
            code_sha=CONTRACT["SOURCE_COMMIT"],
        )
        active_pools_after = threadpool_info()
        if any(int(item.get("num_threads", 0) or 0) > int(CONTRACT["THREAD_LIMIT"]) for item in active_pools_after):
            raise RuntimeError({"thread_pool_limit_failed_after": active_pools_after})
except Exception as exc:
    failure = {
        "schema_version": 1,
        "experiment_id": "E010",
        "status": "FAILED",
        "source_commit": CONTRACT["SOURCE_COMMIT"],
        "started_at_utc": started_at.isoformat(),
        "failed_at_utc": _dt.datetime.now(_dt.timezone.utc).isoformat(),
        "error_type": type(exc).__name__,
        "error": str(exc),
        "traceback": traceback.format_exc(),
    }
    (Path("/kaggle/working") / CONTRACT["RUN_RECEIPT_FILENAME"]).write_text(
        json.dumps(failure, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    raise
ended_at = _dt.datetime.now(_dt.timezone.utc)
print(json.dumps({
    "status": summary["status"],
    "reported_candidate": summary["reported_candidate"],
    "selected_candidate": summary["selected_candidate"],
    "bank_oracle_rmse": summary["bank_oracle_metrics"]["rmse"],
    "reported_rmse": summary["candidate_metrics"][summary["reported_candidate"]]["rmse"],
    "wall_seconds": (ended_at - started_at).total_seconds(),
}, indent=2, sort_keys=True))


In [ ]:
import hashlib
import json
import platform
import shutil
import zipfile
from pathlib import Path, PurePosixPath

WORKING = Path("/kaggle/working")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(16 * 1024 * 1024)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def _safe(name: str) -> str:
    raw = str(name).replace("\\", "/")
    raw_parts = raw.split("/")
    value = PurePosixPath(raw)
    if value.is_absolute() or not raw or any(part in {"", ".", ".."} for part in raw_parts) or not value.parts:
        raise RuntimeError(f"unsafe output member {name!r}")
    return value.as_posix()

output_files = []
for base, prefix in ((RESULT_DIR, "results"), (ARTIFACT_DIR, "artifacts")):
    for path in sorted(base.rglob("*")):
        if path.is_file():
            output_files.append({
                "path": f"{prefix}/{path.relative_to(base).as_posix()}",
                "source": str(path),
                "bytes": path.stat().st_size,
                "sha256": _sha256(path),
            })
if not output_files or not any(item["path"] == "results/summary.json" for item in output_files):
    raise RuntimeError("E010 result set is incomplete")

manifest = {
    "schema_version": 1,
    "experiment_id": "E010",
    "source_commit": CONTRACT["SOURCE_COMMIT"],
    "config_sha256": CONTRACT["CONFIG_SHA256"],
    "input_bundle_sha256": CONTRACT["BUNDLE_SHA256"],
    "files": [{key: item[key] for key in ("path", "bytes", "sha256")} for item in output_files],
}
manifest_path = WORKING / CONTRACT["OUTPUT_MANIFEST_FILENAME"]
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")

archive_path = WORKING / CONTRACT["RESULT_ARCHIVE_FILENAME"]
temporary = archive_path.with_suffix(".zip.tmp")
temporary.unlink(missing_ok=True)
with zipfile.ZipFile(temporary, "w", allowZip64=True) as archive:
    for item in output_files:
        info = zipfile.ZipInfo(_safe(item["path"]), date_time=(1980, 1, 1, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        info.create_system = 3
        info.external_attr = 0o100644 << 16
        with archive.open(info, "w", force_zip64=True) as destination, Path(item["source"]).open("rb") as source:
            shutil.copyfileobj(source, destination, length=16 * 1024 * 1024)
    info = zipfile.ZipInfo(CONTRACT["OUTPUT_MANIFEST_FILENAME"], date_time=(1980, 1, 1, 0, 0, 0))
    info.compress_type = zipfile.ZIP_DEFLATED
    info.create_system = 3
    info.external_attr = 0o100644 << 16
    archive.writestr(info, manifest_path.read_bytes())
temporary.replace(archive_path)

receipt = {
    "schema_version": 1,
    "experiment_id": "E010",
    "status": "COMPLETE",
    "source_commit": CONTRACT["SOURCE_COMMIT"],
    "config_sha256": CONTRACT["CONFIG_SHA256"],
    "input_bundle": {"sha256": CONTRACT["BUNDLE_SHA256"], "bytes": CONTRACT["BUNDLE_BYTES"]},
    "dataset": {"ref": CONTRACT["DATASET_REF"], "version": CONTRACT["DATASET_VERSION"]},
    "started_at_utc": started_at.isoformat(),
    "ended_at_utc": ended_at.isoformat(),
    "wall_seconds": (ended_at - started_at).total_seconds(),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "thread_limit": CONTRACT["THREAD_LIMIT"],
    "thread_pools_before": active_pools_before,
    "thread_pools_after": active_pools_after,
    "internet_expected": False,
    "accelerator_expected": "CPU",
    "summary": {
        "status": summary["status"],
        "selected_candidate": summary["selected_candidate"],
        "reported_candidate": summary["reported_candidate"],
        "eligible_candidates": summary["eligible_candidates"],
        "candidate_count": summary["candidate_count"],
        "advanced_families": summary["advanced_families"],
        "last_known_rmse": summary["baseline_metrics"]["rmse"],
        "e006_rmse": summary["e006_metrics"]["rmse"],
        "bank_oracle_rmse": summary["bank_oracle_metrics"]["rmse"],
        "reported_rmse": summary["candidate_metrics"][summary["reported_candidate"]]["rmse"],
        "all_controls_pass": all(bool(item["pass"]) for item in summary["controls"].values()),
        "runtime": summary["runtime"],
    },
    "output_manifest": {"filename": manifest_path.name, "bytes": manifest_path.stat().st_size, "sha256": _sha256(manifest_path)},
    "result_archive": {"filename": archive_path.name, "bytes": archive_path.stat().st_size, "sha256": _sha256(archive_path)},
    "submission_created": False,
    "submission_made": False,
}
receipt_path = WORKING / CONTRACT["RUN_RECEIPT_FILENAME"]
receipt_path.write_text(json.dumps(receipt, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(receipt, indent=2, sort_keys=True))
